# 08_01_温度_top-k_top-p_min-p

コード：`ch08/blocks/08_01_温度_top-k_top-p_min-p.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 温度・top-k・top-p・min-p を、鋭い文脈と平坦な文脈で比べる
# ロジットは説明用に手で決めた値（実際のモデルの出力ではない）
import numpy as np

SHARP = {"東京": 8.0, "京都": 4.5, "どこ": 4.0, "大阪": 3.0, "東京都": 2.8,
         "江戸": 2.0, "日本": 1.5, "奈良": 1.0, "何": 0.8, "首都": 0.5}
verbs = ("歩いた 言った 笑った 座った うなずいた 目を閉じた 立ち上がった 微笑んだ "
         "息を吐いた 首を振った 部屋を出た 窓を開けた 振り返った 待った 答えた "
         "本を閉じた 席を立った 語り始めた 涙を流した 考えた 手を挙げた 目を開けた "
         "耳を澄ませた 去った 茶を飲んだ 祈った 眠った 歌った 泣いた 頭を下げた "
         "扉を叩いた 空を見た 手紙を読んだ 煙草を消した 筆を置いた 灯りを消した "
         "笑い出した 叫んだ 走った 踊った").split()
FLAT = {w: 3.0 - 0.075 * i for i, w in enumerate(verbs)}     # 40 語がなだらかに並ぶ

def sample_dist(logits, T=1.0, top_k=None, top_p=None, min_p=None):
    words = list(logits)
    z = np.array([logits[w] for w in words]) / T          # 温度で割る
    p = np.exp(z - z.max()); p /= p.sum()
    keep = np.ones(len(p), bool)
    order = np.argsort(-p)
    if top_k is not None:                                  # 上位 k 個
        keep[order[top_k:]] = False
    if top_p is not None:                                  # 累積が p に届くまで
        cum = np.cumsum(p[order])
        keep[order[np.searchsorted(cum, top_p) + 1:]] = False
    if min_p is not None:                                  # 1 位の min_p 倍未満を落とす
        keep &= p >= min_p * p.max()
    q = np.where(keep, p, 0.0); q /= q.sum()               # 再正規化
    return words, p, q

def summary(name, logits, **kw):
    words, p, q = sample_dist(logits, **kw)
    alive = q > 0
    H = -(p * np.log(p)).sum()
    top = sorted(zip(q, words), reverse=True)[:3]
    print(f"{name:6s} {str(kw):42s} 生き残り={alive.sum():2d}語  1位={q.max():.3f}  "
          f"exp(H)={np.exp(H):5.1f}  切り捨て={p[~alive].sum():.3f}  上位="+ " ".join(f"{w}:{x:.3f}" for x, w in top))

for name, lg in (("鋭い", SHARP), ("平坦", FLAT)):
    summary(name, lg)
    summary(name, lg, top_p=0.9)
    summary(name, lg, T=1.5, top_p=0.9)
    summary(name, lg, top_k=5)
    summary(name, lg, T=1.6)
    summary(name, lg, T=1.6, min_p=0.05)

# T=0.1 で 1,000 回引く
rng = np.random.default_rng(0)
for name, lg in (("鋭い", SHARP), ("平坦", FLAT)):
    words, _, q = sample_dist(lg, T=0.1)
    draws = rng.choice(len(words), size=1000, p=q)
    vals, cnt = np.unique(draws, return_counts=True)
    print(name, "T=0.1 ×1000:", {words[v]: int(c) for v, c in zip(vals, cnt)})
